In [ ]:
# %% [markdown]
# # Optimization Log Copilot Prototype
# 
# **Project:** Using Large Language Models to Analyze Optimization Solver Logs  
# **Course:** Introduction to Generative AI  
# **Student:** Jeffrey Ginn
# 
# This prototype notebook demonstrates a lightweight pipeline for:
# 1. loading sample optimization solver logs,
# 2. parsing key events from the logs,
# 3. converting the logs into a structured representation,
# 4. generating a rule-based diagnostic summary,
# 5. preparing prompts for a future LLM-based version.

# %% [markdown]
# ## 1. Imports

# %%
import re
import json
from dataclasses import dataclass, asdict
from typing import Any, Dict, List, Optional

import pandas as pd

# %% [markdown]
# ## 2. Sample Solver Logs
# 
# These are small synthetic examples designed to mimic the kinds of information
# that appear in optimization solver logs. They are good enough for a midpoint
# prototype and can later be replaced with real logs from benchmark runs.

# %%
sample_logs = {
    "weak_root_gap": """
Optimize a model with 500 rows, 1200 columns and 4200 nonzeros
Presolve removed 20 rows and 35 columns
Presolve time: 0.01s
Root relaxation: objective 124.300000e+00, 180 iterations, 0.03 seconds
    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time
     0     0  124.30000    0   18          -  124.30000      -     -    0s
H    0     0                     210.0000000  124.30000  40.8%     -    0s
     0     0  130.12000    0   15  210.00000  130.12000  38.0%     -    0s
Cutting planes:
  Gomory: 5
  MIR: 3
  Flow cover: 2
Explored 15432 nodes (80211 simplex iterations) in 18.42 seconds
Best objective 2.100000000000e+02, best bound 1.681000000000e+02, gap 19.95%
""",
    "healthy_run": """
Optimize a model with 800 rows, 2000 columns and 9000 nonzeros
Presolve removed 120 rows and 200 columns
Presolve time: 0.02s
Root relaxation: objective 510.000000e+00, 220 iterations, 0.05 seconds
    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time
     0     0  510.00000    0    4          -  510.00000      -     -    0s
H    0     0                     540.0000000  510.00000   5.6%     -    0s
     0     0  523.50000    0    2  540.00000  523.50000   3.1%     -    0s
Cutting planes:
  Gomory: 1
  MIR: 1
Explored 82 nodes (1440 simplex iterations) in 0.91 seconds
Best objective 5.400000000000e+02, best bound 5.360000000000e+02, gap 0.74%
""",
    "heavy_cuts_no_progress": """
Optimize a model with 950 rows, 3400 columns and 15000 nonzeros
Presolve removed 10 rows and 12 columns
Presolve time: 0.03s
Root relaxation: objective 88.000000e+00, 900 iterations, 0.15 seconds
    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time
     0     0   88.00000    0   33          -   88.00000      -     -    0s
H    0     0                     180.0000000   88.00000  51.1%     -    0s
     0     0   90.20000    0   31  180.00000   90.20000  49.9%     -    0s
Cutting planes:
  Gomory: 24
  MIR: 18
  Flow cover: 9
  Zero half: 6
Explored 30120 nodes (201553 simplex iterations) in 32.80 seconds
Best objective 1.800000000000e+02, best bound 1.050000000000e+02, gap 41.67%
"""
}

# %% [markdown]
# ## 3. Define a Structured Representation

# %%
@dataclass
class SolverLogFeatures:
    name: str
    rows: Optional[int] = None
    cols: Optional[int] = None
    nonzeros: Optional[int] = None
    presolve_rows_removed: Optional[int] = None
    presolve_cols_removed: Optional[int] = None
    presolve_time_sec: Optional[float] = None
    root_relax_obj: Optional[float] = None
    root_relax_iterations: Optional[int] = None
    root_relax_time_sec: Optional[float] = None
    incumbent: Optional[float] = None
    best_bound: Optional[float] = None
    final_gap_pct: Optional[float] = None
    explored_nodes: Optional[int] = None
    simplex_iterations: Optional[int] = None
    runtime_sec: Optional[float] = None
    cuts: Optional[Dict[str, int]] = None

# %% [markdown]
# ## 4. Parsing Functions

# %%
def _search(pattern: str, text: str, flags: int = 0):
    return re.search(pattern, text, flags)


def parse_model_size(text: str) -> Dict[str, Optional[int]]:
    m = _search(r"Optimize a model with (\d+) rows, (\d+) columns and (\d+) nonzeros", text)
    if not m:
        return {"rows": None, "cols": None, "nonzeros": None}
    return {
        "rows": int(m.group(1)),
        "cols": int(m.group(2)),
        "nonzeros": int(m.group(3)),
    }


def parse_presolve(text: str) -> Dict[str, Optional[float]]:
    removed = _search(r"Presolve removed (\d+) rows and (\d+) columns", text)
    t = _search(r"Presolve time:\s*([0-9.]+)s", text)
    return {
        "presolve_rows_removed": int(removed.group(1)) if removed else None,
        "presolve_cols_removed": int(removed.group(2)) if removed else None,
        "presolve_time_sec": float(t.group(1)) if t else None,
    }


def parse_root_relaxation(text: str) -> Dict[str, Optional[float]]:
    m = _search(
        r"Root relaxation: objective\s+([0-9.eE+\-]+),\s+(\d+) iterations,\s+([0-9.]+) seconds",
        text,
    )
    if not m:
        return {
            "root_relax_obj": None,
            "root_relax_iterations": None,
            "root_relax_time_sec": None,
        }
    return {
        "root_relax_obj": float(m.group(1)),
        "root_relax_iterations": int(m.group(2)),
        "root_relax_time_sec": float(m.group(3)),
    }


def parse_cuts(text: str) -> Dict[str, int]:
    cuts: Dict[str, int] = {}
    block = _search(r"Cutting planes:\n(.*?)(?:Explored|$)", text, flags=re.DOTALL)
    if not block:
        return cuts
    for line in block.group(1).splitlines():
        line = line.strip()
        if not line or ":" not in line:
            continue
        name, value = line.split(":", 1)
        value = value.strip()
        if value.isdigit():
            cuts[name.strip()] = int(value)
    return cuts


def parse_final_stats(text: str) -> Dict[str, Optional[float]]:
    explored = _search(r"Explored (\d+) nodes \((\d+) simplex iterations\) in ([0-9.]+) seconds", text)
    final = _search(
        r"Best objective\s+([0-9.eE+\-]+), best bound\s+([0-9.eE+\-]+), gap\s+([0-9.]+)%",
        text,
    )
    return {
        "explored_nodes": int(explored.group(1)) if explored else None,
        "simplex_iterations": int(explored.group(2)) if explored else None,
        "runtime_sec": float(explored.group(3)) if explored else None,
        "incumbent": float(final.group(1)) if final else None,
        "best_bound": float(final.group(2)) if final else None,
        "final_gap_pct": float(final.group(3)) if final else None,
    }


def parse_solver_log(name: str, text: str) -> SolverLogFeatures:
    data: Dict[str, Any] = {"name": name}
    data.update(parse_model_size(text))
    data.update(parse_presolve(text))
    data.update(parse_root_relaxation(text))
    data.update(parse_final_stats(text))
    data["cuts"] = parse_cuts(text)
    return SolverLogFeatures(**data)

# %% [markdown]
# ## 5. Parse the Sample Logs

# %%
parsed_logs: List[SolverLogFeatures] = [
    parse_solver_log(name, text) for name, text in sample_logs.items()
]

parsed_logs

# %% [markdown]
# ## 6. Convert to a Table

# %%
def flatten_features(features: SolverLogFeatures) -> Dict[str, Any]:
    row = asdict(features)
    cuts = row.pop("cuts") or {}
    for cut_name, cut_count in cuts.items():
        row[f"cut_{cut_name.lower().replace(' ', '_')}"] = cut_count
    row["total_cuts"] = sum(cuts.values())
    return row


df = pd.DataFrame([flatten_features(x) for x in parsed_logs]).fillna(0)
df

# %% [markdown]
# ## 7. Rule-Based Diagnostic Layer
# 
# Before integrating an LLM, it is useful to build a transparent baseline. This
# also makes the notebook more convincing for a midpoint submission because it
# shows concrete logic and measurable outputs.

# %%
def diagnose_log(features: SolverLogFeatures) -> Dict[str, Any]:
    findings: List[str] = []
    recommendations: List[str] = []

    gap = features.final_gap_pct if features.final_gap_pct is not None else 0.0
    nodes = features.explored_nodes if features.explored_nodes is not None else 0
    total_cuts = sum((features.cuts or {}).values())
    root_iters = features.root_relax_iterations if features.root_relax_iterations is not None else 0

    if gap >= 25:
        findings.append("The final optimality gap is large, suggesting that the LP relaxation may be weak or that progress stalled during search.")
        recommendations.append("Consider strengthening the formulation or adding stronger valid inequalities to tighten the relaxation.")
    elif gap <= 2:
        findings.append("The final optimality gap is small, indicating that the solver made strong progress toward optimality.")

    if nodes >= 10000:
        findings.append("The solver explored a large number of branch-and-bound nodes, which may indicate weak branching decisions or a difficult combinatorial structure.")
        recommendations.append("Consider testing stronger branching rules, better primal heuristics, or symmetry-breaking constraints.")
    elif nodes <= 100:
        findings.append("The solver explored relatively few nodes, which suggests that the formulation and search strategy were effective on this instance.")

    if total_cuts >= 20 and gap >= 20:
        findings.append("Many cutting planes were generated, but the remaining gap is still large. This suggests that cut activity alone did not sufficiently improve the bound.")
        recommendations.append("Investigate whether the generated cuts are effective, and consider problem-specific cuts or reformulation.")
    elif 0 < total_cuts < 5:
        findings.append("Only a small number of cuts were generated, which may mean the instance was already relatively well behaved or that cut opportunities were limited.")

    if root_iters >= 800:
        findings.append("The root relaxation required many simplex iterations, which may indicate a numerically challenging or highly degenerate LP relaxation.")
        recommendations.append("Check scaling, coefficient ranges, and formulation tightness to reduce numerical difficulty.")

    if not recommendations:
        recommendations.append("This run appears relatively healthy. A useful next step would be comparing it against alternate parameter settings or formulations.")

    summary = " ".join(findings)
    return {
        "name": features.name,
        "findings": findings,
        "recommendations": recommendations,
        "summary": summary,
    }


diagnostics = [diagnose_log(x) for x in parsed_logs]
diagnostics

# %% [markdown]
# ## 8. Display Human-Readable Reports

# %%
for report in diagnostics:
    print(f"=== {report['name']} ===")
    print("Summary:")
    print(report["summary"])
    print("\nRecommendations:")
    for rec in report["recommendations"]:
        print(f"- {rec}")
    print()

# %% [markdown]
# ## 9. Export a Structured JSON Example
# 
# This is useful for showing how raw logs can be transformed into a format that
# can be consumed by an LLM or stored for evaluation.

# %%
structured_example = asdict(parsed_logs[0])
print(json.dumps(structured_example, indent=2))

# %% [markdown]
# ## 10. Prompt Template for Future LLM Integration
# 
# This cell does not call an external API. Instead, it prepares the prompt that
# would be sent to an LLM in a later version of the project.

# %%
def build_prompt(features: SolverLogFeatures) -> str:
    return f"""
You are an expert in mathematical optimization and solver diagnostics.
Analyze the following structured solver log features and provide:
1. a short summary of solver behavior,
2. likely performance issues,
3. actionable recommendations.

Structured log data:
{json.dumps(asdict(features), indent=2)}

Focus on issues such as weak LP relaxation, heavy branching, ineffective cuts,
numerical difficulty, and solver progress toward optimality.
""".strip()

print(build_prompt(parsed_logs[0]))

# %% [markdown]
# ## 11. Placeholder Experimental Plan
# 
# These placeholders show how the notebook will later support the paper's
# experimental section.

# %%
experimental_plan = {
    "research_questions": [
        "Can an LLM generate accurate summaries of optimization solver logs?",
        "Can an LLM identify likely causes of poor solver performance?",
        "Can an LLM generate useful solver recommendations?",
    ],
    "planned_evaluations": [
        "Compare generated summaries against expert-written summaries.",
        "Measure whether diagnoses match expected solver behaviors.",
        "Collect qualitative ratings on usefulness and correctness.",
    ],
    "future_dataset": "Replace synthetic logs with benchmark logs from real solver runs.",
}

experimental_plan

# %% [markdown]
# ## 12. Next Steps
# 
# - Replace synthetic logs with real solver logs from benchmark instances.
# - Expand the parser to support more solver event types.
# - Integrate an LLM API or local model for real generated summaries.
# - Build an evaluation set with expert labels.
# - Add comparison between raw-log prompting and structured-log prompting.
